<a href="https://colab.research.google.com/github/Khaisekabeer/vision_Transformers/blob/main/ViT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

# -----------------------------
# Patch Embedding
# -----------------------------
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_channels=3, embed_dim=128):
        super().__init__()
        self.patch_size = patch_size
        self.n_patches = (img_size // patch_size) ** 2

        self.proj = nn.Conv2d(
            in_channels,
            embed_dim,
            kernel_size=patch_size,
            stride=patch_size
        )

    def forward(self, x):
        x = self.proj(x)  # (B, embed_dim, H/P, W/P)
        x = x.flatten(2)  # (B, embed_dim, N)
        x = x.transpose(1, 2)  # (B, N, embed_dim)
        return x


# -----------------------------
# Multi-Head Attention
# -----------------------------
class Attention(nn.Module):
    def __init__(self, dim, heads=4):
        super().__init__()
        self.heads = heads
        self.scale = dim ** -0.5

        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x):
        B, N, C = x.shape

        qkv = self.qkv(x).reshape(B, N, 3, self.heads, C // self.heads)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        return x


# -----------------------------
# Transformer Encoder Block
# -----------------------------
class TransformerBlock(nn.Module):
    def __init__(self, dim, heads, mlp_dim):
        super().__init__()

        self.norm1 = nn.LayerNorm(dim)
        self.attn = Attention(dim, heads)

        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, dim)
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


# -----------------------------
# Vision Transformer
# -----------------------------
class VisionTransformer(nn.Module):
    def __init__(
        self,
        img_size=32,
        patch_size=4,
        in_channels=3,
        num_classes=10,
        embed_dim=128,
        depth=6,
        heads=4,
        mlp_dim=256
    ):
        super().__init__()

        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        num_patches = self.patch_embed.n_patches

        # CLS token
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))

        # Positional embedding
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches + 1, embed_dim))

        # Transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, heads, mlp_dim)
            for _ in range(depth)
        ])

        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        B = x.shape[0]

        x = self.patch_embed(x)

        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)

        x = x + self.pos_embed

        for block in self.blocks:
            x = block(x)

        x = self.norm(x)
        cls_output = x[:, 0]

        return self.head(cls_output)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])


trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform
)

trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=128, shuffle=True
)

testset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform
)

testloader = torch.utils.data.DataLoader(
    testset, batch_size=128, shuffle=False
)


model = VisionTransformer().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 1e-4)

# -----------------------------
# Training Loop
# -----------------------------
for epoch in range(20):
    model.train()
    total_loss = 0

    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(trainloader)
    print(f"Epoch {epoch+1}, Avg Loss: {avg_loss:.4f}")

    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
      for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Epoch {epoch+1}, Accuracy: {accuracy:.2f}%")

100%|██████████| 170M/170M [00:03<00:00, 43.5MB/s]


Epoch 1, Avg Loss: 1.9925
Epoch 1, Accuracy: 34.85%
Epoch 2, Avg Loss: 1.7009
Epoch 2, Accuracy: 44.15%
Epoch 3, Avg Loss: 1.5056
Epoch 3, Accuracy: 48.88%
Epoch 4, Avg Loss: 1.3999
Epoch 4, Accuracy: 51.40%
Epoch 5, Avg Loss: 1.3272
Epoch 5, Accuracy: 52.04%
Epoch 6, Avg Loss: 1.2674
Epoch 6, Accuracy: 54.29%
Epoch 7, Avg Loss: 1.2136
Epoch 7, Accuracy: 54.39%
Epoch 8, Avg Loss: 1.1658
Epoch 8, Accuracy: 56.50%
Epoch 9, Avg Loss: 1.1235
Epoch 9, Accuracy: 57.14%
Epoch 10, Avg Loss: 1.0791
Epoch 10, Accuracy: 57.33%
Epoch 11, Avg Loss: 1.0397
Epoch 11, Accuracy: 58.04%
Epoch 12, Avg Loss: 1.0058
Epoch 12, Accuracy: 58.84%
Epoch 13, Avg Loss: 0.9669
Epoch 13, Accuracy: 59.42%
Epoch 14, Avg Loss: 0.9309
Epoch 14, Accuracy: 58.51%
Epoch 15, Avg Loss: 0.8962
Epoch 15, Accuracy: 59.65%
Epoch 16, Avg Loss: 0.8592
Epoch 16, Accuracy: 59.17%
Epoch 17, Avg Loss: 0.8236
Epoch 17, Accuracy: 60.46%
Epoch 18, Avg Loss: 0.7899
Epoch 18, Accuracy: 59.46%
Epoch 19, Avg Loss: 0.7526
Epoch 19, Accuracy:

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

# -----------------------------
# Patch Embedding
# -----------------------------
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_channels=3, embed_dim=128):
        super().__init__()
        self.n_patches = (img_size // patch_size) ** 2

        self.proj = nn.Conv2d(
            in_channels,
            embed_dim,
            kernel_size=patch_size,
            stride=patch_size
        )

    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2)
        x = x.transpose(1, 2)
        return x


# -----------------------------
# Attention
# -----------------------------
class Attention(nn.Module):
    def __init__(self, dim, heads=4):
        super().__init__()
        self.heads = heads
        self.scale = dim ** -0.5

        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x):
        B, N, C = x.shape

        qkv = self.qkv(x).reshape(B, N, 3, self.heads, C // self.heads)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        return x


# -----------------------------
# Transformer Block (UPDATED)
# -----------------------------
class TransformerBlock(nn.Module):
    def __init__(self, dim, heads, mlp_dim):
        super().__init__()

        self.norm1 = nn.LayerNorm(dim)
        self.attn = Attention(dim, heads)

        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, dim)
        )

        self.dropout = nn.Dropout(0.1)   # ✅ Added

    def forward(self, x):
        x = x + self.dropout(self.attn(self.norm1(x)))   # ✅ Added
        x = x + self.dropout(self.mlp(self.norm2(x)))    # ✅ Added
        return x


# -----------------------------
# Vision Transformer
# -----------------------------
class VisionTransformer(nn.Module):
    def __init__(
        self,
        img_size=32,
        patch_size=4,
        in_channels=3,
        num_classes=10,
        embed_dim=128,
        depth=6,
        heads=4,
        mlp_dim=256
    ):
        super().__init__()

        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        num_patches = self.patch_embed.n_patches

        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches + 1, embed_dim))

        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, heads, mlp_dim)
            for _ in range(depth)
        ])

        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        B = x.shape[0]

        x = self.patch_embed(x)

        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)

        x = x + self.pos_embed

        for block in self.blocks:
            x = block(x)

        x = self.norm(x)
        cls_output = x[:, 0]

        return self.head(cls_output)


# -----------------------------
# Device
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# -----------------------------
# Data Augmentation (UPDATED)
# -----------------------------
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])


# -----------------------------
# Dataset
# -----------------------------
trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform
)

trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=128, shuffle=True
)

testset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform
)

testloader = torch.utils.data.DataLoader(
    testset, batch_size=128, shuffle=False
)


# -----------------------------
# Model
# -----------------------------
model = VisionTransformer().to(device)

criterion = nn.CrossEntropyLoss()

# ✅ AdamW optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

# ✅ Scheduler
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)


# -----------------------------
# Training Loop
# -----------------------------
for epoch in range(30):   # ✅ Increased epochs
    model.train()
    total_loss = 0

    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(trainloader)

    # -----------------------------
    # Evaluation
    # -----------------------------
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total

    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%")

    # ✅ Step scheduler
    scheduler.step()

Epoch 1, Loss: 2.0380, Accuracy: 30.80%
Epoch 2, Loss: 1.8679, Accuracy: 35.98%
Epoch 3, Loss: 1.7265, Accuracy: 40.44%
Epoch 4, Loss: 1.6363, Accuracy: 44.12%
Epoch 5, Loss: 1.5769, Accuracy: 45.93%
Epoch 6, Loss: 1.5223, Accuracy: 46.65%
Epoch 7, Loss: 1.4984, Accuracy: 47.55%
Epoch 8, Loss: 1.4767, Accuracy: 47.85%
Epoch 9, Loss: 1.4550, Accuracy: 47.47%
Epoch 10, Loss: 1.4374, Accuracy: 49.37%
Epoch 11, Loss: 1.4186, Accuracy: 50.20%
Epoch 12, Loss: 1.4108, Accuracy: 49.79%
Epoch 13, Loss: 1.4007, Accuracy: 50.14%
Epoch 14, Loss: 1.3919, Accuracy: 50.71%
Epoch 15, Loss: 1.3870, Accuracy: 50.92%
Epoch 16, Loss: 1.3764, Accuracy: 51.44%
Epoch 17, Loss: 1.3758, Accuracy: 51.24%
Epoch 18, Loss: 1.3694, Accuracy: 51.59%
Epoch 19, Loss: 1.3686, Accuracy: 51.79%
Epoch 20, Loss: 1.3641, Accuracy: 52.05%
Epoch 21, Loss: 1.3606, Accuracy: 51.56%
Epoch 22, Loss: 1.3575, Accuracy: 51.99%
Epoch 23, Loss: 1.3581, Accuracy: 51.92%
Epoch 24, Loss: 1.3546, Accuracy: 52.09%
Epoch 25, Loss: 1.3514, A

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

# -----------------------------
# Patch Embedding
# -----------------------------
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_channels=3, embed_dim=128):
        super().__init__()
        self.n_patches = (img_size // patch_size) ** 2

        self.proj = nn.Conv2d(
            in_channels,
            embed_dim,
            kernel_size=patch_size,
            stride=patch_size
        )

    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2)
        x = x.transpose(1, 2)
        return x


# -----------------------------
# Attention
# -----------------------------
class Attention(nn.Module):
    def __init__(self, dim, heads=4):
        super().__init__()
        self.heads = heads
        self.scale = dim ** -0.5

        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)

        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        B, N, C = x.shape

        qkv = self.qkv(x).reshape(B, N, 3, self.heads, C // self.heads)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.dropout(attn)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.dropout(x)
        return x


# -----------------------------
# Transformer Block (UPDATED)
# -----------------------------
class TransformerBlock(nn.Module):
    def __init__(self, dim, heads, mlp_dim):
        super().__init__()

        self.norm1 = nn.LayerNorm(dim)
        self.attn = Attention(dim, heads)

        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, dim)
        )

        self.dropout = nn.Dropout(0.1)   # ✅ Added

    def forward(self, x):
        x = x + self.dropout(self.attn(self.norm1(x)))   # ✅ Added
        x = x + self.dropout(self.mlp(self.norm2(x)))    # ✅ Added
        return x


# -----------------------------
# Vision Transformer
# -----------------------------
class VisionTransformer(nn.Module):
    def __init__(
        self,
        img_size=32,
        patch_size=4,
        in_channels=3,
        num_classes=10,
        embed_dim=128,
        depth=6,
        heads=4,
        mlp_dim=256
    ):
        super().__init__()

        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        num_patches = self.patch_embed.n_patches

        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches + 1, embed_dim))

        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, heads, mlp_dim)
            for _ in range(depth)
        ])

        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        B = x.shape[0]

        x = self.patch_embed(x)

        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)

        x = x + self.pos_embed

        for block in self.blocks:
            x = block(x)

        x = self.norm(x)
        cls_output = x[:, 0]

        return self.head(cls_output)


# -----------------------------
# Device
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# -----------------------------
# Data Augmentation (UPDATED)
# -----------------------------
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])


# -----------------------------
# Dataset
# -----------------------------
trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform
)

trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=128, shuffle=True
)

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

testset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=test_transform
)

testloader = torch.utils.data.DataLoader(
    testset, batch_size=128, shuffle=False
)


# -----------------------------
# Model
# -----------------------------
model = VisionTransformer(
    embed_dim=256,
    depth=8,
    heads=8,
    mlp_dim=512
).to(device)

criterion = nn.CrossEntropyLoss()

# ✅ AdamW optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr = 3e-4, weight_decay=1e-4)

# ✅ Scheduler
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)


# -----------------------------
# Training Loop
# -----------------------------
for epoch in range(30):   # ✅ Increased epochs
    model.train()
    total_loss = 0

    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(trainloader)

    # -----------------------------
    # Evaluation
    # -----------------------------
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total

    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%")

    # ✅ Step scheduler
    scheduler.step()

Epoch 1, Loss: 1.9404, Accuracy: 41.99%
Epoch 2, Loss: 1.5910, Accuracy: 49.31%
Epoch 3, Loss: 1.4300, Accuracy: 53.90%
Epoch 4, Loss: 1.3437, Accuracy: 55.57%
Epoch 5, Loss: 1.2768, Accuracy: 58.46%
Epoch 6, Loss: 1.1799, Accuracy: 59.86%
Epoch 7, Loss: 1.1342, Accuracy: 61.81%
Epoch 8, Loss: 1.1022, Accuracy: 62.35%
Epoch 9, Loss: 1.0718, Accuracy: 62.32%
Epoch 10, Loss: 1.0391, Accuracy: 63.18%
Epoch 11, Loss: 0.9832, Accuracy: 65.48%
Epoch 12, Loss: 0.9574, Accuracy: 65.70%
Epoch 13, Loss: 0.9439, Accuracy: 65.74%
Epoch 14, Loss: 0.9270, Accuracy: 66.70%
Epoch 15, Loss: 0.9136, Accuracy: 67.24%
Epoch 16, Loss: 0.8827, Accuracy: 67.41%
Epoch 17, Loss: 0.8691, Accuracy: 67.95%
Epoch 18, Loss: 0.8606, Accuracy: 68.33%
Epoch 19, Loss: 0.8567, Accuracy: 68.49%
Epoch 20, Loss: 0.8465, Accuracy: 67.90%
Epoch 21, Loss: 0.8263, Accuracy: 68.90%
Epoch 22, Loss: 0.8231, Accuracy: 68.97%
Epoch 23, Loss: 0.8218, Accuracy: 69.20%
Epoch 24, Loss: 0.8172, Accuracy: 69.50%
Epoch 25, Loss: 0.8125, A

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

# -----------------------------
# Patch Embedding (EDGE)
# -----------------------------
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=32, patch_size=8, in_channels=3, embed_dim=128):
        super().__init__()
        self.n_patches = (img_size // patch_size) ** 2

        self.proj = nn.Conv2d(
            in_channels,
            embed_dim,
            kernel_size=patch_size,
            stride=patch_size
        )

    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2)
        x = x.transpose(1, 2)
        return x


# -----------------------------
# Lightweight Attention
# -----------------------------
class Attention(nn.Module):
    def __init__(self, dim, heads=2):
        super().__init__()
        self.heads = heads
        self.scale = dim ** -0.5

        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)

        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        B, N, C = x.shape

        qkv = self.qkv(x).reshape(B, N, 3, self.heads, C // self.heads)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.dropout(attn)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.dropout(x)
        return x


# -----------------------------
# Transformer Block
# -----------------------------
class TransformerBlock(nn.Module):
    def __init__(self, dim, heads, mlp_dim):
        super().__init__()

        self.norm1 = nn.LayerNorm(dim)
        self.attn = Attention(dim, heads)

        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, dim)
        )

        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        x = x + self.dropout(self.attn(self.norm1(x)))
        x = x + self.dropout(self.mlp(self.norm2(x)))
        return x


# -----------------------------
# Vision Transformer (EDGE)
# -----------------------------
class VisionTransformer(nn.Module):
    def __init__(
        self,
        img_size=32,
        patch_size=8,     # ✅ larger patches (less compute)
        in_channels=3,
        num_classes=10,
        embed_dim=128,    # ✅ smaller embedding
        depth=4,          # ✅ fewer layers
        heads=2,          # ✅ fewer heads
        mlp_dim=256
    ):
        super().__init__()

        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        num_patches = self.patch_embed.n_patches

        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches + 1, embed_dim))

        self.pos_drop = nn.Dropout(0.1)

        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, heads, mlp_dim)
            for _ in range(depth)
        ])

        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        B = x.shape[0]

        x = self.patch_embed(x)

        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)

        x = x + self.pos_embed
        x = self.pos_drop(x)

        for block in self.blocks:
            x = block(x)

        x = self.norm(x)
        cls_output = x[:, 0]

        return self.head(cls_output)


# -----------------------------
# Device
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# -----------------------------
# Data
# -----------------------------
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=train_transform
)

testset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=test_transform
)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True)
testloader = torch.utils.data.DataLoader(testset, batch_size=128, shuffle=False)


# -----------------------------
# Model
# -----------------------------
model = VisionTransformer().to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40)


# -----------------------------
# Training Loop
# -----------------------------
for epoch in range(40):
    model.train()
    total_loss = 0

    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(trainloader)

    # -----------------------------
    # Evaluation
    # -----------------------------
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total

    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%")

    scheduler.step()


# -----------------------------
# Quantization (EDGE DEPLOY)
# -----------------------------
quantized_model = torch.quantization.quantize_dynamic(
    model, {nn.Linear}, dtype=torch.qint8
)

print("Quantized model ready for edge deployment 🚀")

Epoch 1, Loss: 2.0419, Accuracy: 37.43%
Epoch 2, Loss: 1.8521, Accuracy: 41.92%
Epoch 3, Loss: 1.7786, Accuracy: 45.73%
Epoch 4, Loss: 1.7374, Accuracy: 47.72%
Epoch 5, Loss: 1.7079, Accuracy: 48.84%
Epoch 6, Loss: 1.6810, Accuracy: 48.29%
Epoch 7, Loss: 1.6589, Accuracy: 49.66%
Epoch 8, Loss: 1.6408, Accuracy: 51.24%
Epoch 9, Loss: 1.6236, Accuracy: 52.59%
Epoch 10, Loss: 1.6077, Accuracy: 53.01%
Epoch 11, Loss: 1.5914, Accuracy: 51.91%
Epoch 12, Loss: 1.5798, Accuracy: 52.78%
Epoch 13, Loss: 1.5681, Accuracy: 53.36%
Epoch 14, Loss: 1.5514, Accuracy: 54.10%
Epoch 15, Loss: 1.5458, Accuracy: 54.08%
Epoch 16, Loss: 1.5310, Accuracy: 54.48%
Epoch 17, Loss: 1.5232, Accuracy: 55.39%
Epoch 18, Loss: 1.5125, Accuracy: 55.48%
Epoch 19, Loss: 1.5077, Accuracy: 56.73%
Epoch 20, Loss: 1.4965, Accuracy: 56.78%
Epoch 21, Loss: 1.4886, Accuracy: 56.70%
Epoch 22, Loss: 1.4861, Accuracy: 57.61%
Epoch 23, Loss: 1.4798, Accuracy: 57.68%
Epoch 24, Loss: 1.4737, Accuracy: 57.92%
Epoch 25, Loss: 1.4644, A

/tmp/ipykernel_15716/891155638.py:231: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

# -----------------------------
# Patch Embedding (EDGE)
# -----------------------------
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=32, patch_size=8, in_channels=3, embed_dim=128):
        super().__init__()
        self.n_patches = (img_size // patch_size) ** 2

        self.proj = nn.Conv2d(
            in_channels,
            embed_dim,
            kernel_size=patch_size,
            stride=patch_size
        )

    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2)
        x = x.transpose(1, 2)
        return x


# -----------------------------
# Lightweight Attention
# -----------------------------
class Attention(nn.Module):
    def __init__(self, dim, heads=2):
        super().__init__()
        self.heads = heads
        self.scale = dim ** -0.5

        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)

        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        B, N, C = x.shape

        qkv = self.qkv(x).reshape(B, N, 3, self.heads, C // self.heads)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.dropout(attn)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.dropout(x)
        return x


# -----------------------------
# Transformer Block
# -----------------------------
class TransformerBlock(nn.Module):
    def __init__(self, dim, heads, mlp_dim):
        super().__init__()

        self.norm1 = nn.LayerNorm(dim)
        self.attn = Attention(dim, heads)

        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, dim)
        )

        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        x = x + self.dropout(self.attn(self.norm1(x)))
        x = x + self.dropout(self.mlp(self.norm2(x)))
        return x


# -----------------------------
# Vision Transformer (EDGE)
# -----------------------------
class VisionTransformer(nn.Module):
    def __init__(
        self,
        img_size=32,
        patch_size=8,     # ✅ larger patches (less compute)
        in_channels=3,
        num_classes=10,
        embed_dim=160,    # ✅ smaller embedding
        depth=4,          # ✅ fewer layers
        heads=2,          # ✅ fewer heads
        mlp_dim=320
    ):
        super().__init__()

        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        num_patches = self.patch_embed.n_patches

        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches + 1, embed_dim))

        self.pos_drop = nn.Dropout(0.1)

        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, heads, mlp_dim)
            for _ in range(depth)
        ])

        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        B = x.shape[0]

        x = self.patch_embed(x)

        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)

        x = x + self.pos_embed
        x = self.pos_drop(x)

        for block in self.blocks:
            x = block(x)

        x = self.norm(x)
        cls_output = x[:, 0]

        return self.head(cls_output)


# -----------------------------
# Device
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# -----------------------------
# Data
# -----------------------------
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=train_transform
)

testset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=test_transform
)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True)
testloader = torch.utils.data.DataLoader(testset, batch_size=128, shuffle=False)


# -----------------------------
# Model
# -----------------------------
model = VisionTransformer().to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40)


# -----------------------------
# Training Loop
# -----------------------------
for epoch in range(40):
    model.train()
    total_loss = 0

    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(trainloader)

    # -----------------------------
    # Evaluation
    # -----------------------------
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total

    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%")

    scheduler.step()


# -----------------------------
# Quantization (EDGE DEPLOY)
# -----------------------------
import torchao.quantization as tq

quantized_model = tq.quantize_dynamic(
    model, {nn.Linear}, dtype=torch.qint8
)
print("Quantized model ready for edge deployment 🚀")

Epoch 1, Loss: 2.0245, Accuracy: 38.83%
Epoch 2, Loss: 1.8444, Accuracy: 44.11%
Epoch 3, Loss: 1.7730, Accuracy: 45.74%
Epoch 4, Loss: 1.7311, Accuracy: 46.93%
Epoch 5, Loss: 1.6950, Accuracy: 49.03%
Epoch 6, Loss: 1.6653, Accuracy: 50.94%
Epoch 7, Loss: 1.6426, Accuracy: 50.38%
Epoch 8, Loss: 1.6215, Accuracy: 52.09%
Epoch 9, Loss: 1.6000, Accuracy: 52.63%
Epoch 10, Loss: 1.5867, Accuracy: 52.35%
Epoch 11, Loss: 1.5688, Accuracy: 53.80%
Epoch 12, Loss: 1.5543, Accuracy: 53.97%
Epoch 13, Loss: 1.5350, Accuracy: 54.65%
Epoch 14, Loss: 1.5243, Accuracy: 55.82%
Epoch 15, Loss: 1.5134, Accuracy: 55.86%
Epoch 16, Loss: 1.5021, Accuracy: 55.97%
Epoch 17, Loss: 1.4898, Accuracy: 56.69%
Epoch 18, Loss: 1.4810, Accuracy: 57.24%
Epoch 19, Loss: 1.4712, Accuracy: 58.48%
Epoch 20, Loss: 1.4617, Accuracy: 58.03%
Epoch 21, Loss: 1.4548, Accuracy: 58.48%
Epoch 22, Loss: 1.4452, Accuracy: 59.00%
Epoch 23, Loss: 1.4373, Accuracy: 59.59%
Epoch 24, Loss: 1.4334, Accuracy: 59.71%
Epoch 25, Loss: 1.4261, A

AttributeError: module 'torchao.quantization' has no attribute 'quantize_dynamic'

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

# -----------------------------
# Patch Embedding (EDGE)
# -----------------------------
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=32, patch_size=8, in_channels=3, embed_dim=128):
        super().__init__()
        self.n_patches = (img_size // patch_size) ** 2

        self.proj = nn.Conv2d(
            in_channels,
            embed_dim,
            kernel_size=patch_size,
            stride=patch_size
        )

    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2)
        x = x.transpose(1, 2)
        return x


# -----------------------------
# Lightweight Attention
# -----------------------------
class Attention(nn.Module):
    def __init__(self, dim, heads=2):
        super().__init__()
        self.heads = heads
        self.scale = dim ** -0.5

        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)

        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        B, N, C = x.shape

        qkv = self.qkv(x).reshape(B, N, 3, self.heads, C // self.heads)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.dropout(attn)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.dropout(x)
        return x


# -----------------------------
# Transformer Block
# -----------------------------
class TransformerBlock(nn.Module):
    def __init__(self, dim, heads, mlp_dim):
        super().__init__()

        self.norm1 = nn.LayerNorm(dim)
        self.attn = Attention(dim, heads)

        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, dim)
        )

        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        x = x + self.dropout(self.attn(self.norm1(x)))
        x = x + self.dropout(self.mlp(self.norm2(x)))
        return x


# -----------------------------
# Vision Transformer (EDGE)
# -----------------------------
class VisionTransformer(nn.Module):
    def __init__(
        self,
        img_size=32,
        patch_size=8,     # ✅ larger patches (less compute)
        in_channels=3,
        num_classes=10,
        embed_dim=160,    # ✅ smaller embedding
        depth=4,          # ✅ fewer layers
        heads=2,          # ✅ fewer heads
        mlp_dim=320
    ):
        super().__init__()

        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        num_patches = self.patch_embed.n_patches

        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches + 1, embed_dim))

        self.pos_drop = nn.Dropout(0.1)

        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, heads, mlp_dim)
            for _ in range(depth)
        ])

        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        B = x.shape[0]

        x = self.patch_embed(x)

        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)

        x = x + self.pos_embed
        x = self.pos_drop(x)

        for block in self.blocks:
            x = block(x)

        x = self.norm(x)
        cls_output = x[:, 0]

        return self.head(cls_output)


# -----------------------------
# Device
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# -----------------------------
# Data
# -----------------------------
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=train_transform
)

testset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=test_transform
)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True)
testloader = torch.utils.data.DataLoader(testset, batch_size=128, shuffle=False)


# -----------------------------
# Model
# -----------------------------
model = VisionTransformer().to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40)


# -----------------------------
# Training Loop
# -----------------------------
for epoch in range(40):
    model.train()
    total_loss = 0

    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(trainloader)

    # -----------------------------
    # Evaluation
    # -----------------------------
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total

    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%")

    scheduler.step()


# -----------------------------
# Quantization (EDGE DEPLOY)
# -----------------------------
quantized_model = torch.quantization.quantize_dynamic(
    model, {nn.Linear}, dtype=torch.qint8
)
print("Quantized model ready for edge deployment 🚀")

Epoch 1, Loss: 2.0246, Accuracy: 39.63%
Epoch 2, Loss: 1.8384, Accuracy: 43.56%
Epoch 3, Loss: 1.7728, Accuracy: 46.32%
Epoch 4, Loss: 1.7294, Accuracy: 48.06%
Epoch 5, Loss: 1.6995, Accuracy: 48.90%
Epoch 6, Loss: 1.6696, Accuracy: 50.94%
Epoch 7, Loss: 1.6483, Accuracy: 50.07%
Epoch 8, Loss: 1.6330, Accuracy: 51.73%
Epoch 9, Loss: 1.6110, Accuracy: 51.69%
Epoch 10, Loss: 1.5925, Accuracy: 53.55%
Epoch 11, Loss: 1.5744, Accuracy: 53.96%
Epoch 12, Loss: 1.5627, Accuracy: 53.28%
Epoch 13, Loss: 1.5468, Accuracy: 53.40%
Epoch 14, Loss: 1.5390, Accuracy: 54.75%
Epoch 15, Loss: 1.5221, Accuracy: 55.40%
Epoch 16, Loss: 1.5113, Accuracy: 55.30%
Epoch 17, Loss: 1.5006, Accuracy: 56.46%
Epoch 18, Loss: 1.4882, Accuracy: 57.12%
Epoch 19, Loss: 1.4752, Accuracy: 56.62%
Epoch 20, Loss: 1.4724, Accuracy: 57.84%
Epoch 21, Loss: 1.4648, Accuracy: 57.97%
Epoch 22, Loss: 1.4534, Accuracy: 58.48%
Epoch 23, Loss: 1.4447, Accuracy: 58.44%
Epoch 24, Loss: 1.4359, Accuracy: 59.44%
Epoch 25, Loss: 1.4323, A

/tmp/ipykernel_15716/995650536.py:231: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


In [ ]:
import time

def measure_time(model, loader):
    model.eval()
    start = time.time()

    with torch.no_grad():
        for images, _ in loader:
            images = images.to(device)
            _ = model(images)

    end = time.time()
    return end - start


# 🔴 Define full model
full_model = VisionTransformer(
    patch_size=4,
    embed_dim=256,
    depth=8,
    heads=8,
    mlp_dim=512
).to(device)

# 🟢 Edge model already exists → model
# ⚡ Quantized model already exists → quantized_model

print("Full model time:", measure_time(full_model, testloader))
print("Edge model time:", measure_time(model, testloader))
print("Quantized model time:", measure_time(quantized_model, testloader))

Full model time: 3.0921542644500732
Edge model time: 3.1151840686798096


NotImplementedError: Could not run 'quantized::linear_dynamic' with arguments from the 'CUDA' backend. This could be because the operator doesn't exist for this backend, or was omitted during the selective/custom build process (if using custom build). If you are a Facebook employee using PyTorch on mobile, please visit https://fburl.com/ptmfixes for possible resolutions. 'quantized::linear_dynamic' is only available for these backends: [CPU, Meta, BackendSelect, Python, FuncTorchDynamicLayerBackMode, Functionalize, Named, Conjugate, Negative, ZeroTensor, ADInplaceOrView, AutogradOther, AutogradCPU, AutogradCUDA, AutogradXLA, AutogradMPS, AutogradXPU, AutogradHPU, AutogradLazy, AutogradMTIA, AutogradMAIA, AutogradPrivateUse1, AutogradMeta, Tracer, AutocastCPU, AutocastMTIA, AutocastMAIA, AutocastXPU, AutocastMPS, AutocastCUDA, FuncTorchBatched, BatchedNestedTensor, FuncTorchVmapMode, Batched, VmapMode, FuncTorchGradWrapper, PythonTLSSnapshot, FuncTorchDynamicLayerFrontMode, PreDispatch,

In [ ]:
def measure_time_cpu(model, loader):
    model.eval()
    model.to("cpu")

    start = time.time()

    with torch.no_grad():
        for images, _ in loader:
            images = images.to("cpu")
            _ = model(images)

    end = time.time()
    return end - start


print("Full model time:", measure_time_cpu(full_model, testloader))
print("Edge model time:", measure_time_cpu(model, testloader))
print("Quantized model time:", measure_time_cpu(quantized_model, testloader))

Full model time: 76.72898817062378
Edge model time: 7.006926536560059


RuntimeError: apply_dynamic is not implemented for this packed parameter type

In [ ]:
# Move edge model to CPU first
model_cpu = model.to("cpu")

# Apply quantization
quantized_model = torch.quantization.quantize_dynamic(
    model_cpu, {nn.Linear}, dtype=torch.qint8
)
print("Full model time:", measure_time_cpu(full_model, testloader))
print("Edge model time:", measure_time_cpu(model, testloader))
print("Quantized model time:", measure_time_cpu(quantized_model, testloader))

/tmp/ipykernel_15716/3817179848.py:5: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


Full model time: 77.42892003059387
Edge model time: 6.5263755321502686
Quantized model time: 6.706660509109497


In [ ]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, pred = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (pred == labels).sum().item()

    return 100 * correct / total


# 🔴 Full model
print("Full:", evaluate(full_model, testloader))

# 🟢 Edge model
print("Edge:", evaluate(model, testloader))

# ⚡ Quantized model
print("Quantized:", evaluate(quantized_model, testloader))

RuntimeError: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

In [ ]:
device = torch.device("cpu")   # 🔥 IMPORTANT

def evaluate(model, loader):
    model.eval()
    model.to(device)

    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, pred = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (pred == labels).sum().item()

    return 100 * correct / total


print("Full:", evaluate(full_model, testloader))
print("Edge:", evaluate(model, testloader))
print("Quantized:", evaluate(quantized_model, testloader))

Full: 9.35
Edge: 61.06
Quantized: 61.1


In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import time

# -----------------------------
# DEVICE
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# DATA
# -----------------------------
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=train_transform
)
testset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=test_transform
)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True)
testloader = torch.utils.data.DataLoader(testset, batch_size=128, shuffle=False)

# -----------------------------
# MODEL COMPONENTS
# -----------------------------
class PatchEmbedding(nn.Module):
    def __init__(self, img_size, patch_size, embed_dim):
        super().__init__()
        self.proj = nn.Conv2d(3, embed_dim, patch_size, patch_size)

    def forward(self, x):
        x = self.proj(x)
        return x.flatten(2).transpose(1, 2)

class Attention(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.heads = heads
        self.scale = dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.heads, C // self.heads)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.proj(x)

class Block(nn.Module):
    def __init__(self, dim, heads, mlp_dim):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = Attention(dim, heads)
        self.norm2 = nn.LayerNorm(dim)

        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, dim)
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

class VisionTransformer(nn.Module):
    def __init__(self, patch_size, embed_dim, depth, heads, mlp_dim):
        super().__init__()

        self.patch_embed = PatchEmbedding(32, patch_size, embed_dim)
        num_patches = (32 // patch_size) ** 2

        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches + 1, embed_dim))

        self.blocks = nn.ModuleList([
            Block(embed_dim, heads, mlp_dim) for _ in range(depth)
        ])

        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, 10)

    def forward(self, x):
        B = x.size(0)

        x = self.patch_embed(x)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls, x), dim=1)

        x = x + self.pos_embed

        for blk in self.blocks:
            x = blk(x)

        x = self.norm(x)
        return self.head(x[:, 0])

# -----------------------------
# TRAIN FUNCTION
# -----------------------------
def train_model(model, epochs):
    model.to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for images, labels in trainloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}, Loss: {total_loss/len(trainloader):.4f}")

# -----------------------------
# EVALUATE FUNCTION
# -----------------------------
def evaluate(model):
    model.eval()
    model.to("cpu")

    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in testloader:
            images = images.to("cpu")
            outputs = model(images)
            _, pred = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (pred == labels).sum().item()

    return 100 * correct / total

# -----------------------------
# TIME FUNCTION
# -----------------------------
def measure_time(model):
    model.eval()
    model.to("cpu")

    start = time.time()
    with torch.no_grad():
        for images, _ in testloader:
            images = images.to("cpu")
            _ = model(images)
    return time.time() - start

# -----------------------------
# TRAIN FULL MODEL
# -----------------------------
full_model = VisionTransformer(4, 256, 8, 8, 512)
train_model(full_model, 40)
torch.save(full_model.state_dict(), "full_model.pth")

# -----------------------------
# TRAIN EDGE MODEL
# -----------------------------
edge_model = VisionTransformer(8, 160, 4, 2, 320)
train_model(edge_model, 40)
torch.save(edge_model.state_dict(), "edge_model.pth")

# -----------------------------
# LOAD MODELS (optional reuse)
# -----------------------------
# full_model.load_state_dict(torch.load("full_model.pth"))
# edge_model.load_state_dict(torch.load("edge_model.pth"))

# -----------------------------
# QUANTIZATION (CPU ONLY)
# -----------------------------
edge_model_cpu = edge_model.to("cpu")
quantized_model = torch.quantization.quantize_dynamic(
    edge_model_cpu, {nn.Linear}, dtype=torch.qint8
)

# -----------------------------
# RESULTS
# -----------------------------
print("\n===== ACCURACY =====")
print("Full:", evaluate(full_model))
print("Edge:", evaluate(edge_model))
print("Quantized:", evaluate(quantized_model))

print("\n===== TIME =====")
print("Full:", measure_time(full_model))
print("Edge:", measure_time(edge_model))
print("Quantized:", measure_time(quantized_model))

100%|██████████| 170M/170M [00:04<00:00, 37.4MB/s]


Epoch 1, Loss: 1.9646
Epoch 2, Loss: 1.6903
Epoch 3, Loss: 1.5711
Epoch 4, Loss: 1.4916
Epoch 5, Loss: 1.4398
Epoch 6, Loss: 1.3874
Epoch 7, Loss: 1.3455
Epoch 8, Loss: 1.3042
Epoch 9, Loss: 1.2675
Epoch 10, Loss: 1.2333
Epoch 11, Loss: 1.2061
Epoch 12, Loss: 1.1740
Epoch 13, Loss: 1.1498
Epoch 14, Loss: 1.1242
Epoch 15, Loss: 1.0968
Epoch 16, Loss: 1.0768
Epoch 17, Loss: 1.0504
Epoch 18, Loss: 1.0268
Epoch 19, Loss: 1.0077
Epoch 20, Loss: 0.9820
Epoch 21, Loss: 0.9603
Epoch 22, Loss: 0.9363
Epoch 23, Loss: 0.9184
Epoch 24, Loss: 0.8896
Epoch 25, Loss: 0.8716
Epoch 26, Loss: 0.8442
Epoch 27, Loss: 0.8300
Epoch 28, Loss: 0.8059
Epoch 29, Loss: 0.7831
Epoch 30, Loss: 0.7696
Epoch 31, Loss: 0.7543
Epoch 32, Loss: 0.7322
Epoch 33, Loss: 0.7260
Epoch 34, Loss: 0.7075
Epoch 35, Loss: 0.6937
Epoch 36, Loss: 0.6807
Epoch 37, Loss: 0.6732
Epoch 38, Loss: 0.6641
Epoch 39, Loss: 0.6537
Epoch 40, Loss: 0.6453
Epoch 1, Loss: 1.9652
Epoch 2, Loss: 1.7771
Epoch 3, Loss: 1.7005
Epoch 4, Loss: 1.6479
E

/tmp/ipykernel_995/3713036591.py:203: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


Full: 74.84
Edge: 67.32
Quantized: 67.26

===== TIME =====
Full: 79.51098489761353
Edge: 6.816614151000977
Quantized: 5.840660810470581
